# Data transformations and adjustments: lab notebook

**Statistical Modelling and Forecasting, Statistics background for forecasting**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

### 1. Box-Cox by hand and with scipy

In [ ]:
# 1. Box-Cox by hand and with scipy
y = 49.0
print("lambda=0.5 :", (y ** 0.5 - 1) / 0.5, "| lambda=0 :", round(np.log(y), 4))
w = 6
print("inverse of w=6 (lambda=0.5):", (0.5 * w + 1) ** 2)
print("scipy boxcox(49, 0.5):", stats.boxcox(np.array([49.0, 1.0]), lmbda=0.5)[0])

### 2. Estimating lambda: real GDP grows exponentially, so a log-like lambda is chosen

In [ ]:
# 2. Estimating lambda: real GDP grows exponentially, so a log-like lambda is chosen
gdp = sm.datasets.macrodata.load_pandas().data["realgdp"].values
transformed, lam = stats.boxcox(gdp)
print("maximum-likelihood lambda for real GDP:", round(lam, 2))
cpi = sm.datasets.macrodata.load_pandas().data["cpi"].values
print("lambda for CPI:", round(stats.boxcox(cpi)[1], 2))

### 3. Variance stabilisation: spread of quarterly changes, early against late

In [ ]:
# 3. Variance stabilisation: spread of quarterly changes, early against late
d_raw = np.diff(gdp)
d_log = np.diff(np.log(gdp))
h = len(d_raw) // 2
print("raw changes   : std first half", round(d_raw[:h].std(), 1), "| second half", round(d_raw[h:].std(), 1))
print("log changes   : std first half", round(d_log[:h].std(), 4), "| second half", round(d_log[h:].std(), 4))

### 4. Adjustments: real values, per-day values, per-head values

In [ ]:
# 4. Adjustments: real values, per-day values, per-head values
nominal, price_index = 5400, 120
print("real value:", nominal / price_index * 100)
total, days = 372, 31
print("per day:", total / days)
df = sm.datasets.macrodata.load_pandas().data
real_gdp_per_pop = (df["realgdp"] / df["pop"]).iloc[[0, -1]].round(2).tolist()
print("real GDP per person (first, last):", real_gdp_per_pop)

### 5. Back-transforming a forecast from a log model gives a median forecast

In [ ]:
# 5. Back-transforming a forecast from a log model gives a median forecast
x = np.log(gdp)
slope, intercept = np.polyfit(np.arange(len(x)), x, 1)
f_log = intercept + slope * len(x)
print("forecast of log GDP:", round(f_log, 4), "-> back-transformed:", round(np.exp(f_log), 1))

## Expected output

Running all cells above should print:

```
lambda=0.5 : 12.0 | lambda=0 : 3.8918
inverse of w=6 (lambda=0.5): 16.0
scipy boxcox(49, 0.5): 12.0
maximum-likelihood lambda for real GDP: 0.16
lambda for CPI: 0.35
raw changes   : std first half 52.5 | second half 63.9
log changes   : std first half 0.0108 | second half 0.006
real value: 4500.0
per day: 12.0
real GDP per person (first, last): [15.3, 42.17]
forecast of log GDP: 9.5869 -> back-transformed: 14573.3
```

## Practice exercises

Try these in the cells below.

1. With the multiplicative seasonal series, move λ from 1 to 0. Find the λ that makes the seasonal swing the same size at the start and the end.
2. Choose the nominal series and tick "Apply the adjustment". How does the slope change?
3. Choose the monthly totals series and apply the per-day adjustment. Does the zig-zag disappear?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
